In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# Updated Label Mapping
label_map = {
    'benign': 0, 'adware': 1, 'backdoor': 2, 'banker': 3, 'dropper': 4,
    'fileinjector': 5, 'NoCategory': 6, 'PUA': 7, 'Ransomware': 8,
    'Riskware': 9, 'SMS': 10, 'Scareware': 11, 'Spy': 12, 'Trojan': 13, 'Zeroday': 14
}

# Load datasets (Replace 'malware_multiclass.csv' with your actual multiclass filename)
benign_df = pd.read_csv("processedData/benign.csv")
mal_df = pd.read_csv("processedData/malware_multiclass.csv") 

# Reset columns to ensure consistency before concatenation
benign_df.columns = range(benign_df.shape[1])
mal_df.columns = range(mal_df.shape[1])

# Concatenate datasets
df = pd.concat([benign_df, mal_df], axis=0, ignore_index=True)

# Ensure feature 9504 contains the integer values 0-14 according to label_map
print("Value counts for multiclass labels:")
print(df[9504].value_counts())

In [ ]:



X = df.drop(columns=[9504, 1])  # also drop hash column (1)
y = df[9504]

# Compute correlation of each feature with label
correlations = X.corrwith(y)

# Sort by absolute correlation
corr_sorted = correlations.abs().sort_values(ascending=False)

print(corr_sorted.head(200))

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Use ANOVA F-test to find the best features for multiclass separation
selector = SelectKBest(score_func=f_classif, k=20)
X_selected = selector.fit_transform(X, y)
top_features = X.columns[selector.get_support()].tolist()

print("Selected Multiclass Feature Indices:", top_features)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_reduced = X[top_features]
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)

# Model initialization remains the same
model = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
target_names = list(label_map.keys())

print("Multiclass Accuracy:", model.score(X_test, y_test))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
import numpy as np

y_train_shuffled = np.random.permutation(y_train)
model.fit(X_train, y_train_shuffled)
print("Accuracy with shuffled labels:", model.score(X_test, y_test))

In [ ]:

import shap
from sklearn.ensemble import RandomForestClassifier

# sample only 200 rows
X_sample = X_train.sample(10000)

# train a smaller RF just for SHAP
rf_shap = RandomForestClassifier(
    n_estimators=50,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_shap.fit(X_train, y_train)

explainer = shap.TreeExplainer(rf_shap)


explainer = shap.TreeExplainer(rf_shap)

shap_values = explainer(X_sample)  # only malware class
malicious_shap_values = shap_values [:, :, 1]

shap.summary_plot(malicious_shap_values, X_sample)


In [ ]:
shap.summary_plot(shap_values[:, :, 1], X_sample, plot_type="bar")

In [ ]:
sample = X_test.iloc[0:10]
pred = model.predict(sample)
# 2. Iterate through the predictions

explainer = shap.TreeExplainer(model)
shap_explanations = explainer(sample, check_additivity=False)

for i, app_pred in enumerate(pred):
    status = "Malware" if app_pred == 1 else "Benign"
    
    # Get the SHAP values for this specific app (class 1: Malicious)
    # We look for the highest positive contributions for Malware
    # and highest negative contributions for Benign
    current_shap_values = shap_explanations.values[i, :, 1]
    feature_names = sample.columns
    
    # Sort features by their impact (absolute SHAP value)
    top_indices = np.argsort(np.abs(current_shap_values))[-3:][::-1]
    
    reasons = []
    for idx in top_indices:
        val = current_shap_values[idx]
        feature_name = feature_names[idx]
        direction = "contributed to Malware risk" if val > 0 else "suggested Benign behavior"
        reasons.append(f"{feature_name} ({direction} risk by {abs(val):.4f})%")
        

    print(f"App {i}: {status} detected.")
    print(f"   Reasoning: {', '.join(reasons)}\n")